# 中国主要城市生活成本与性价比分析
# Cost of Living & Affordability Analysis in Chinese Cities

> **项目背景**：对于求职者和应届生来说，选择去哪个城市发展是重要决策。本项目从生活成本、收入水平、买房压力等多个维度，对中国主要城市进行系统性对比分析，帮助大家做出更理性的选择。

> **分析维度**：
> 1. 各城市生活成本排名与构成
> 2. 收入 vs 支出：哪个城市性价比最高？
> 3. 重点城市多维度雷达图对比
> 4. 买房压力测算：攒钱多少年能买房？
> 5. 不同人群的城市选择建议

> **关键词**：生活成本、性价比、城市对比、可视化分析

## 0. 环境配置 / Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

# 配置
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.float_format', lambda x: '%.0f' % x)
pd.set_option('display.max_columns', 20)

# 导入工具函数
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from utils import (calculate_disposable_income, plot_radar_chart, 
                   plot_cost_breakdown, plot_salary_vs_cost,
                   calculate_cost_index, plot_affordability_ranking)

print('Libraries imported successfully.')

## 1. 数据加载与概览 / Data Overview

In [ ]:
# ===== 数据说明 / Data Source =====
# 基于2025年公开数据整理，来源包括：智联招聘薪酬报告、国家统计局、中指研究院、贝壳研究院等
# 工资：税前平均招聘月薪（智联招聘/薪酬网2025年数据）
# 房租：一居室整租月租估算（基于各城市租金均价，约40㎡标准）
# 房价：新建商品住宅均价（中指研究院/聚汇数据2025年数据）
# 注：为城市平均估算值，具体因区域/行业/个人差异较大，仅供参考

cities_data = [
    # 一线城市
    {'city': '北京', 'tier': '一线', 'avg_salary_before_tax': 13400, 'rent_1br': 4200, 'food_monthly': 3000, 'transport_monthly': 300, 'utilities_monthly': 400, 'other_monthly': 1500, 'house_price_per_sqm': 56500},
    {'city': '上海', 'tier': '一线', 'avg_salary_before_tax': 13700, 'rent_1br': 4000, 'food_monthly': 3200, 'transport_monthly': 350, 'utilities_monthly': 420, 'other_monthly': 1600, 'house_price_per_sqm': 65000},
    {'city': '深圳', 'tier': '一线', 'avg_salary_before_tax': 12800, 'rent_1br': 3700, 'food_monthly': 2800, 'transport_monthly': 280, 'utilities_monthly': 450, 'other_monthly': 1400, 'house_price_per_sqm': 65400},
    {'city': '广州', 'tier': '一线', 'avg_salary_before_tax': 11200, 'rent_1br': 2300, 'food_monthly': 2500, 'transport_monthly': 250, 'utilities_monthly': 380, 'other_monthly': 1200, 'house_price_per_sqm': 30000},
    
    # 新一线城市
    {'city': '杭州', 'tier': '新一线', 'avg_salary_before_tax': 12300, 'rent_1br': 2200, 'food_monthly': 2400, 'transport_monthly': 220, 'utilities_monthly': 350, 'other_monthly': 1100, 'house_price_per_sqm': 32900},
    {'city': '成都', 'tier': '新一线', 'avg_salary_before_tax': 10000, 'rent_1br': 1850, 'food_monthly': 1800, 'transport_monthly': 150, 'utilities_monthly': 280, 'other_monthly': 900, 'house_price_per_sqm': 14700},
    {'city': '武汉', 'tier': '新一线', 'avg_salary_before_tax': 10100, 'rent_1br': 1950, 'food_monthly': 1900, 'transport_monthly': 160, 'utilities_monthly': 290, 'other_monthly': 950, 'house_price_per_sqm': 14500},
    {'city': '南京', 'tier': '新一线', 'avg_salary_before_tax': 10900, 'rent_1br': 2100, 'food_monthly': 2200, 'transport_monthly': 200, 'utilities_monthly': 320, 'other_monthly': 1000, 'house_price_per_sqm': 27500},
    {'city': '西安', 'tier': '新一线', 'avg_salary_before_tax': 8900, 'rent_1br': 1650, 'food_monthly': 1700, 'transport_monthly': 140, 'utilities_monthly': 260, 'other_monthly': 800, 'house_price_per_sqm': 15500},
    {'city': '重庆', 'tier': '新一线', 'avg_salary_before_tax': 8500, 'rent_1br': 1400, 'food_monthly': 1700, 'transport_monthly': 150, 'utilities_monthly': 270, 'other_monthly': 850, 'house_price_per_sqm': 12800},
    {'city': '苏州', 'tier': '新一线', 'avg_salary_before_tax': 10600, 'rent_1br': 2200, 'food_monthly': 2300, 'transport_monthly': 210, 'utilities_monthly': 330, 'other_monthly': 1050, 'house_price_per_sqm': 21000},
    {'city': '长沙', 'tier': '新一线', 'avg_salary_before_tax': 9100, 'rent_1br': 1650, 'food_monthly': 1800, 'transport_monthly': 150, 'utilities_monthly': 270, 'other_monthly': 850, 'house_price_per_sqm': 9800},
    
    # 二线城市
    {'city': '青岛', 'tier': '二线', 'avg_salary_before_tax': 8000, 'rent_1br': 1750, 'food_monthly': 1800, 'transport_monthly': 150, 'utilities_monthly': 280, 'other_monthly': 800, 'house_price_per_sqm': 17500},
    {'city': '大连', 'tier': '二线', 'avg_salary_before_tax': 7400, 'rent_1br': 1550, 'food_monthly': 1700, 'transport_monthly': 130, 'utilities_monthly': 290, 'other_monthly': 750, 'house_price_per_sqm': 14500},
    {'city': '厦门', 'tier': '二线', 'avg_salary_before_tax': 10300, 'rent_1br': 2400, 'food_monthly': 2000, 'transport_monthly': 180, 'utilities_monthly': 300, 'other_monthly': 900, 'house_price_per_sqm': 39000},
]

df = pd.DataFrame(cities_data)
print(f'覆盖城市数量: {len(df)}')
print(f'城市等级分布:')
print(df['tier'].value_counts())

In [ ]:
# 计算税后工资（简化估算：扣除五险一金+个税，约为税前的80%）
df['avg_salary_after_tax'] = (df['avg_salary_before_tax'] * 0.8).round(0)

# 计算月度总支出和可支配收入
df = calculate_disposable_income(
    df, 
    salary_col='avg_salary_after_tax',
    rent_col='rent_1br',
    food_col='food_monthly',
    transport_col='transport_monthly',
    utilities_col='utilities_monthly',
    other_col='other_monthly'
)

# 计算生活成本指数（北京=100）
df = calculate_cost_index(df, reference_city='北京')

# 计算买房所需年限（按90平米，存的钱全用来买房，不考虑贷款利息）
df['house_total_price'] = df['house_price_per_sqm'] * 90
df['years_to_buy_house'] = (df['house_total_price'] / (df['disposable_income'] * 12)).round(1)

df.head()

## 2. 生活成本排名 / Cost of Living Ranking

In [ ]:
# 按月度总支出排序
cost_ranking = df.sort_values('monthly_expenses', ascending=False)[[
    'city', 'tier', 'monthly_expenses', 'cost_index', 'avg_salary_after_tax'
]].reset_index(drop=True)
cost_ranking.index = cost_ranking.index + 1
cost_ranking.columns = ['城市', '等级', '月均支出', '成本指数(北京=100)', '税后月薪']
cost_ranking

In [ ]:
# 可视化：生活成本排名
fig, ax = plt.subplots(figsize=(12, 8))

ranked = df.sort_values('monthly_expenses', ascending=True)
tier_colors = {'一线': '#e74c3c', '新一线': '#f39c12', '二线': '#2ecc71'}
colors = [tier_colors[t] for t in ranked['tier']]

bars = ax.barh(ranked['city'], ranked['monthly_expenses'], color=colors, alpha=0.8)

# 添加数值标签
for bar, val in zip(bars, ranked['monthly_expenses']):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}元', va='center', fontsize=9)

# 添加图例
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=label, alpha=0.8)
                   for label, color in tier_colors.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.xlabel('月均生活支出（元）', fontsize=11)
plt.title('各城市生活成本排名', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 支出结构分析 / Cost Breakdown

In [ ]:
# 选取代表性城市对比
selected_cities = ['北京', '上海', '深圳', '杭州', '成都', '武汉', '长沙', '西安']
cost_cols = ['rent_1br', 'food_monthly', 'transport_monthly', 'utilities_monthly', 'other_monthly']
cost_labels = ['房租', '餐饮', '交通', '水电网', '其他']

# 重命名列用于展示
df_plot = df[df['city'].isin(selected_cities)].copy()
df_plot = df_plot.rename(columns=dict(zip(cost_cols, cost_labels)))

fig = plot_cost_breakdown(df_plot, selected_cities, cost_labels, figsize=(12, 6))
plt.show()

In [ ]:
# 计算房租占收入比例
df['rent_to_income_ratio'] = (df['rent_1br'] / df['avg_salary_after_tax'] * 100).round(1)

rent_ratio = df.sort_values('rent_to_income_ratio', ascending=True)[[
    'city', 'rent_1br', 'avg_salary_after_tax', 'rent_to_income_ratio'
]].reset_index(drop=True)
rent_ratio.columns = ['城市', '月房租', '税后月薪', '房租收入比(%)']
rent_ratio

**💡 洞察**：
- 房租是最大支出项，一线城市占比普遍超过 40%
- 「房租收入比」是衡量生活压力的重要指标，30% 以下相对舒适
- 新一线城市中，成都、长沙、西安的房租压力相对较小

## 4. 重点城市多维度对比 / Radar Chart Comparison

In [ ]:
# 选取4个代表性城市做多维度对比
radar_cities = ['北京', '上海', '成都', '杭州']

# 准备雷达图数据（归一化处理，数值越大成本越高）
radar_df = df[df['city'].isin(radar_cities)].copy()

# 选择维度并归一化（0-100分）
radar_dims = ['rent_1br', 'food_monthly', 'transport_monthly', 
              'house_price_per_sqm', 'avg_salary_after_tax']
radar_labels = ['房租成本', '餐饮成本', '交通成本', '房价水平', '收入水平']

for col in radar_dims:
    max_val = radar_df[col].max()
    radar_df[col + '_norm'] = radar_df[col] / max_val * 100

radar_norm_cols = [col + '_norm' for col in radar_dims]
radar_data = radar_df[['city'] + radar_norm_cols]
radar_data.columns = ['city'] + radar_labels

# 绘制雷达图
fig = plot_radar_chart(radar_data, radar_cities, radar_labels, figsize=(10, 10))
plt.show()

## 5. 收入 vs 支出：哪个城市性价比最高？ / Affordability Analysis

In [ ]:
# 散点图：收入 vs 支出
fig = plot_salary_vs_cost(df, x_col='avg_salary_after_tax', 
                          y_col='monthly_expenses', figsize=(10, 7))
plt.show()

**📊 读图说明**：
- 点在**红色虚线左上方**：支出 > 收入，入不敷出
- 点在**红色虚线右下方**：收入 > 支出，能存到钱
- 点**越靠右下**：收入高、支出低，性价比最高

In [ ]:
# 可支配收入排名（每月能存多少钱）
fig = plot_affordability_ranking(df, metric='disposable_income', top_n=15, figsize=(12, 8))
plt.show()

In [ ]:
# 存款率排名
savings_ranking = df.sort_values('savings_rate', ascending=False)[[
    'city', 'tier', 'avg_salary_after_tax', 'monthly_expenses', 
    'disposable_income', 'savings_rate'
]].reset_index(drop=True)
savings_ranking['savings_rate'] = (savings_ranking['savings_rate'] * 100).round(1)
savings_ranking.index = savings_ranking.index + 1
savings_ranking.columns = ['城市', '等级', '税后月薪', '月均支出', '月存款', '存款率(%)']
savings_ranking

## 6. 买房压力测算 / Housing Affordability

In [ ]:
# 买房年限排名（按90平米计算，全款不贷款）
house_ranking = df.sort_values('years_to_buy_house', ascending=True)[[
    'city', 'tier', 'house_price_per_sqm', 'disposable_income', 'years_to_buy_house'
]].reset_index(drop=True)
house_ranking.index = house_ranking.index + 1
house_ranking.columns = ['城市', '等级', '房价(元/平)', '月存款(元)', '全款买房需(年)']
house_ranking

In [ ]:
# 可视化买房年限
fig, ax = plt.subplots(figsize=(12, 8))

ranked = df.sort_values('years_to_buy_house', ascending=True)
tier_colors = {'一线': '#e74c3c', '新一线': '#f39c12', '二线': '#2ecc71'}
colors = [tier_colors[t] for t in ranked['tier']]

bars = ax.barh(ranked['city'], ranked['years_to_buy_house'], color=colors, alpha=0.8)

for bar, val in zip(bars, ranked['years_to_buy_house']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}年', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=label, alpha=0.8)
                   for label, color in tier_colors.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.xlabel('全款买房所需年数（90平米）', fontsize=11)
plt.title('各城市买房压力排名', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 总结与建议 / Conclusion & Recommendations

### 🏆 性价比城市梯队

**第一梯队（收入高+生活压力适中）**
- 广州、杭州：收入不错，生活成本比北上深低一截

**第二梯队（适合攒钱）**
- 长沙、成都、重庆、西安：收入中等，但房价低、生活成本低，存款率高

**第三梯队（高收入高压力）**
- 北京、上海、深圳：收入最高，但房租和房价压力大，适合打拼不适合定居

### 👥 不同人群的城市选择建议

**刚毕业的年轻人 → 推荐：北上深杭**
- 机会多、成长快，趁年轻积累经验和履历
- 房租可以合租降低压力

**想定居买房 → 推荐：长沙、成都、武汉、西安**
- 房价相对友好，生活压力小
- 新一线城市机会也在增多

**追求工作生活平衡 → 推荐：成都、杭州、广州**
- 收入不错，生活节奏相对没那么快
- 美食多、幸福感高

**搞钱优先 → 推荐：深圳、上海、北京**
- 薪资天花板最高
- 适合短期奋斗攒钱，长期定居需谨慎评估

## 8. 数据说明 / Data Notes

- 以上数据为示例估算值，基于2024年公开数据整理
- 工资为平均水平，具体行业/岗位差异很大（互联网/金融会高很多）
- 房租按一居室整租计算，合租成本会低30%-50%
- 买房计算为全款理想情况，实际可贷款降低压力
- 真实数据收集方法请参考 `data/README.md`

---

*本分析仅供参考，每个人的情况不同，选择城市还需结合行业机会、家庭、个人偏好等综合考虑。*